# 📚 Appendix A: Grokking Simplicity

> *Optionales Notebook — für alle die tiefer in die Code-Architektur einsteigen wollen.*

Dieses Notebook erklärt das **Grokking Simplicity**-Prinzip: Wie man LLM-Code sauber in drei Kategorien aufteilt.

# 📊 Data, Calculations, Actions

## Grokking Simplicity für LLM-Programmierung

Das Buch *Grokking Simplicity* teilt Code in drei Kategorien:

| Kategorie | Was ist das? | Beispiel |
|-----------|-------------|------------------|
| **DATA** | Reine Deklarationen, kein Verhalten | Signatures, Datasets |
| **CALCULATIONS** | Reine Funktionen, gleicher Input = gleicher Output | Metriken |
| **ACTIONS** | Seiteneffekte, I/O, Netzwerk | LLM-Aufrufe |

Warum ist das wichtig? Weil DATA und CALCULATIONS **sofort testbar** sind — ohne API-Key, ohne Wartezeit, ohne Kosten. Nur ACTIONS brauchen das echte Modell.

> 💡 *Wir nutzen hier DSPy als Werkzeug, aber die Konzepte — Evaluation, Metriken, Optimierung — gelten für jedes LLM-Framework.*

In [ ]:
import sys
sys.path.insert(0, ".")
import dspy
from dspy_tasks.tasks import get_task, list_by_tier
from dspy_tasks.data import ClassifySentiment, ExtractEntities, SummarizeText
from dspy_tasks.calculations import sentiment_exact_match, entity_f1, summary_quality
from dspy_tasks.actions import run_baseline
from dspy_tasks.visualize import *

In [ ]:
from dspy_tasks.visualize import diagram

diagram([
    {"label": "DATA", "detail": "Signatures, Datasets", "icon": "📦", "color": "#0078d4"},
    {"label": "CALCULATIONS", "detail": "Metriken (rein, testbar)", "icon": "🧮", "color": "#107c10"},
    {"label": "ACTIONS", "detail": "LLM-Aufrufe (I/O)", "icon": "⚡", "color": "#ca5010"},
], title="Die drei Schichten nach Grokking Simplicity")

## Teil 1: DATA — Signatures als Deklarationen

Eine Signature ist **reines Data** — sie beschreibt was du willst, hat aber kein Verhalten. Du könntest sie als JSON serialisieren und übers Netzwerk schicken. Keine Magie, keine LLM-Aufrufe.

In [ ]:
# A Signature is pure DATA — it declares intent, not implementation
print("ClassifySentiment Felder:")
for name in ClassifySentiment.model_fields:
    print(f"  {name}: {ClassifySentiment.__annotations__.get(name)}")

display_insight("DATA-Prinzip",
    "Signatures sind reine Daten-Deklarationen. Sie beschreiben WAS du willst, nie WIE. "
    "Du könntest sie als JSON serialisieren — sie haben kein Verhalten.")

In [ ]:
task = get_task("sentiment")
examples = task.load_examples()

print(f"Dataset: {len(examples)} examples\n")
for ex in examples[:5]:
    print(f"  Review: {str(ex.review)[:60]}...")
    print(f"  Sentiment: {ex.sentiment}")
    print()

## Teil 2: CALCULATIONS — Metriken als reine Funktionen

Metriken sind **Calculations** — gleicher Input, immer gleicher Output. Du kannst sie mit Fake-Daten testen, ganz ohne API-Key. Sie sind unendlich schnell und perfekt deterministisch.

Das hier ist deine **Spezifikation** — deine "Test-Suite" für KI.

In [ ]:
# CALCULATION — No LLM needed! Instantly testable.
class FakeExample:
    sentiment = "positive"
class FakePrediction:
    sentiment = "positive"
class WrongPrediction:
    sentiment = "negative"

score_correct = sentiment_exact_match(FakeExample(), FakePrediction())
score_wrong = sentiment_exact_match(FakeExample(), WrongPrediction())

icon_correct = "✅" if score_correct else "❌"
icon_wrong = "✅" if score_wrong else "❌"
print(f"{icon_correct} Correct prediction score: {score_correct}")
print(f"{icon_wrong} Wrong prediction score: {score_wrong}")

display_insight("CALCULATION-Prinzip",
    "Metriken sind reine Funktionen. Du kannst sie mit Fake-Daten testen, "
    "kein API-Key nötig. Sie sind unendlich schnell und perfekt deterministisch. "
    "Das hier ist deine Spezifikation — deine 'Test-Suite' für KI.")

In [ ]:
class ExampleEntities:
    entities = "Apple, Google, Microsoft"
class PredEntities:
    entities = "Apple, Microsoft, Amazon"  # Got 2 of 3, added 1 wrong

score = entity_f1(ExampleEntities(), PredEntities())
print(f"📊 Entity F1 score: {score:.3f}")
print(f"   (2 correct out of 3 expected, 1 false positive → F1 reflects both precision and recall)")

## Teil 3: ACTIONS — LLM-Aufrufe als I/O

Jetzt wird's ernst: **Actions** sind der Teil mit Seiteneffekten. Sie rufen echte Modelle auf, brauchen Zeit, kosten Geld, und können fehlschlagen. Wir trennen sie, damit alles andere testbar bleibt.

In [ ]:
import ipywidgets as widgets
from dspy_tasks.config import get_available_models, get_default_model, configure_dspy

AVAILABLE_MODELS = get_available_models()

btn = run_button("Run Sentiment Baseline")
out = widgets.Output()

def on_run(b):
    with out:
        out.clear_output()
        print(f"⏳ Running sentiment task on {MODEL}...")
        result = run_baseline("sentiment", max_eval=10)
        display_score("Baseline Score", result.score)
        print(f"⏱️  {result.elapsed_seconds}s | {result.llm_calls} LLM calls")
        display_results_table(result.individual_scores)

btn.on_click(on_run)
display(btn, out)

In [ ]:
# Run all 3 Tier 1 tasks
task_dd = widgets.Dropdown(
    options=[(t.name, t.id) for t in list_by_tier(1)[:3]],
    description="Task:"
)
btn2 = run_button("Run Task")
out2 = widgets.Output()

def on_run2(b):
    with out2:
        out2.clear_output()
        task = get_task(task_dd.value)
        print(f"⏳ Running {task.name} on {MODEL}...")
        result = run_baseline(task_dd.value, max_eval=8)
        display_score(task.name, result.score)
        print(f"⏱️  {result.elapsed_seconds}s | 💡 {task.teaching_point}")
        display_results_table(result.individual_scores[:5])

btn2.on_click(on_run2)
display(widgets.HBox([task_dd, btn2]), out2)

## 🔴 Moment mal — das LLM liegt auch daneben!

Lass uns genauer hinschauen. Nicht jede Antwort ist richtig. Besonders bei kniffligen Fällen — Sarkasmus, gemischte Gefühle, mehrdeutige Reviews — macht das Modell Fehler.

Das ist kein Bug, das ist normal. **Jedes LLM macht Fehler.** Die Frage ist: wie viele? Und wie messen wir das?

In [ ]:
# Lass uns ein paar knifflige Beispiele ausprobieren
from dspy_tasks.config import configure_dspy
from dspy_tasks.data import ClassifySentiment
import dspy

lm = 
tricky_reviews = [
    ("Super, schon wieder ein Produkt das nach einer Woche kaputt geht. Genau was ich brauchte.", "negative"),
    ("Die Kamera ist ok, aber für den Preis hätte ich mehr erwartet.", "neutral"),
    ("Nicht schlecht, aber auch nicht gut. Es tut was es soll.", "neutral"),
    ("Ich LIEBE es wie laut dieses Gerät ist. Meine Nachbarn auch.", "negative"),
    ("Das beste Geschenk das ich je bekommen habe!", "positive"),
    ("Funktioniert. Mehr kann man dazu nicht sagen.", "neutral"),
    ("Absolute Katastrophe. Nie wieder.", "negative"),
    ("Für Anfänger super, Profis werden enttäuscht sein.", "neutral"),
]

classifier = dspy.Predict(ClassifySentiment)
print("Knifflige Reviews — schafft das Modell das?\n")

correct = 0
total = len(tricky_reviews)
for review, expected in tricky_reviews:
    result = classifier(review=review)
    pred = result.sentiment.strip().lower()
    is_correct = pred == expected
    correct += int(is_correct)
    icon = "✅" if is_correct else "❌"
    print(f"{icon} \"{review[:60]}...\"")
    print(f"   Erwartet: {expected} | Modell: {pred}")
    print()

accuracy = correct / total
print("━" * 50)
print(f"Ergebnis: {correct}/{total} richtig = {accuracy:.0%} Accuracy")
if accuracy < 1.0:
    print(f"👆 Siehst du? {total - correct} Fehler! Das LLM ist NICHT perfekt.")

## 📐 Was ist eigentlich "Accuracy"?

Du hast gerade **Accuracy** gesehen — den einfachsten Weg, Qualität zu messen:

> **Accuracy = Anzahl richtige Antworten / Gesamtzahl Fragen**

- 7 von 10 richtig = **70% Accuracy**
- 10 von 10 richtig = **100% Accuracy**

Das ist deine **Metrik** — eine Zahl die sagt, wie gut (oder schlecht) dein LLM arbeitet. Ohne Metrik bist du blind. Mit Metrik kannst du:
- Verschiedene Modelle vergleichen
- Den Effekt von Prompt-Änderungen messen
- Entscheiden, ob eine Änderung wirklich besser ist

> 💡 **Merke:** Eine Metrik ist wie ein Tachometer — sie zeigt dir die Geschwindigkeit. Ohne Tachometer weisst du nicht, ob du schneller oder langsamer fährst.

In [ ]:
# Das Gleiche mit Industrie-Benchmark-Daten
from dspy_tasks.benchmarks import load_truthfulqa, contains_match

tqa_examples = load_truthfulqa(8)

print("🧪 TruthfulQA — Fragen die LLMs zum Halluzinieren verleiten:\n")

class QASignature(dspy.Signature):
    """Answer the question accurately and truthfully."""
    question = dspy.InputField(desc="A factual question")
    answer = dspy.OutputField(desc="A truthful, accurate answer")

qa_module = dspy.Predict(QASignature)
correct = 0
for ex in tqa_examples:
    pred = qa_module(question=ex.question)
    score = contains_match(ex, pred)
    correct += int(score > 0)
    icon = "✅" if score > 0 else "❌"
    print(f"{icon} Frage: {ex.question}")
    print(f"   Erwartet: {str(ex.answer)[:80]}")
    print(f"   Modell: {str(pred.answer)[:80]}")
    print()

acc = correct / len(tqa_examples)
print("━" * 50)
print(f"TruthfulQA: {correct}/{len(tqa_examples)} = {acc:.0%}")
print("👆 Das sind VALIDIERTE Benchmark-Fragen aus der KI-Forschung!")

## 📚 Weiterführend

Dieses Konzept ist die Grundlage der Code-Architektur in diesem Projekt. Für den Hauptlernpfad (Evaluation → Optimierung → Domain-Tuning → Agenten) starte bei **Notebook 00**.

In [ ]:
display_insight("Das Fundament",
    "Signatures und Metriken sind dein Quellcode — rein, testbar und portabel. "
    "LLM-Aufrufe sind nur die I/O-Schicht. Wenn du das Modell wechselst, "
    "ändern sich nur die ACTIONS. DATA und CALCULATIONS bleiben gleich.",
    icon="🏗️")